# 🕵️‍♂️ SherlockAI — Kaggle GPU Remote Inference Bridge

**Hierarchical Multi-Agent Cybercrime Forensic Reconstruction (H-MACFR)**

This notebook provides the **Tier 2 Remote GPU Microservice** for SherlockAI. It:
1. Loads **`Qwen/Qwen3-8B`** (quantized via 4-bit NF4) on Kaggle Cloud GPU (~5.5 GB VRAM).
2. Mounts fine-tuned PEFT LoRA adapters (`base_cybersecurity`, `memory`, `disk`, `logs`, `registry`, and `network`).
3. Exposes an ISO/IEC 27037 authenticated microservice via FastAPI.
4. Spawns an encrypted **Cloudflare Tunnel (`trycloudflare.com`)** so your local SherlockAI backend can query the GPU seamlessly without port forwarding or public IP requirements.


## Step 1: Verify Kaggle GPU & Install Dependencies

In [ ]:
# Check GPU Availability
!nvidia-smi

# Install Required Inference & Server Libraries
!pip install -q -U torch transformers peft bitsandbytes accelerate fastapi uvicorn pydantic pyngrok nest_asyncio

# Install Cloudflare Tunnel for Free Public HTTPS (no signup or token required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print(" Cloudflared tunnel client installed successfully.")

## Step 2: Locate & Unpack Sherlock Adapters
Upload `sherlock_adapters.zip` as a Kaggle Dataset (e.g. named `sherlock-adapters`) or directly into `/kaggle/working/`.

In [ ]:
import os, sys, zipfile, glob
from pathlib import Path

def find_adapters_root():
    candidate_dirs = [
        "/kaggle/working/sherlock_adapters",
        "/kaggle/working",
        "/content/sherlock_adapters",
        "/content"
    ]
    # Check Kaggle dataset input directories
    for p in glob.glob("/kaggle/input/**/sherlock_adapters", recursive=True):
        if os.path.isdir(p):
            candidate_dirs.insert(0, p)
    for p in glob.glob("/kaggle/input/**", recursive=False):
        if os.path.isdir(p):
            candidate_dirs.append(p)

    # If a zip exists in working directory, extract it
    for zip_candidate in ["/kaggle/working/sherlock_adapters.zip", "/content/sherlock_adapters.zip"]:
        if os.path.exists(zip_candidate):
            print(f"📦 Extracting {zip_candidate} to /kaggle/working/...")
            with zipfile.ZipFile(zip_candidate, 'r') as zf:
                zf.extractall("/kaggle/working")
            candidate_dirs.insert(0, "/kaggle/working/sherlock_adapters")

    # Verify which directory contains the clean adapter folders
    for c in candidate_dirs:
        c_path = Path(c)
        if (c_path / "network" / "adapter_config.json").exists() or (c_path / "base_cybersecurity" / "adapter_config.json").exists():
            return str(c_path)
        if (c_path / "sherlock_adapters" / "network" / "adapter_config.json").exists():
            return str(c_path / "sherlock_adapters")
    return "/kaggle/working/sherlock_adapters"

ADAPTERS_ROOT = find_adapters_root()
print(f"🎯 Adapters Root Directory: {ADAPTERS_ROOT}")

# List discovered adapter folders
expected_domains = ["base_cybersecurity", "network", "memory", "disk", "logs", "registry"]
found_adapters = {}
for dom in expected_domains:
    p = Path(ADAPTERS_ROOT) / dom / "adapter_config.json"
    if p.exists():
        found_adapters[dom] = str(Path(ADAPTERS_ROOT) / dom)
        print(f"  ✅ {dom:20} -> {found_adapters[dom]}")
    else:
        print(f"  ⚠️ {dom:20} -> Not found at {p}")

print(f"Total Verified Adapters: {len(found_adapters)} / {len(expected_domains)}")

## Step 3: Load Qwen3-8B Base Model & Multi-LoRA Adapters

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Foundation Base Model for SherlockAI Domain LoRAs (hidden_size: 4096)
BASE_MODEL_ID = "Qwen/Qwen3-8B"
print(f"[1/4] Loading Tokenizer for {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print(f"[2/4] Loading 4-Bit NF4 Base Model ({BASE_MODEL_ID}) onto GPU...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.eval()

print(f"[3/4] Mounting Fine-Tuned Qwen3 PEFT Adapters...")
peft_model = None
loaded_adapters = []

# Adapters trained with Qwen3-8B base (hidden_size: 4096)
qwen3_domains = ["base_cybersecurity", "memory", "disk", "logs", "registry"]

for domain in qwen3_domains:
    path = found_adapters.get(domain)
    if not path:
        continue
    try:
        if peft_model is None:
            peft_model = PeftModel.from_pretrained(base_model, path, adapter_name=domain)
        else:
            peft_model.load_adapter(path, adapter_name=domain)
        loaded_adapters.append(domain)
        print(f"  ✅ Mounted Qwen3 adapter: {domain}")
    except Exception as e:
        print(f"  ⚠️ Could not mount adapter '{domain}': {e}")

inference_model = peft_model if peft_model is not None else base_model

# 4. Dedicated Network Model (Qwen2.5-7B) for Network Adapter
network_model = None
network_tokenizer = None
NETWORK_BASE_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

if "network" in found_adapters:
    print(f"\n[4/4] Mounting Network Adapter ({NETWORK_BASE_ID})...")
    try:
        net_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        net_base = AutoModelForCausalLM.from_pretrained(
            NETWORK_BASE_ID,
            quantization_config=net_bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        net_base.eval()
        network_model = PeftModel.from_pretrained(net_base, found_adapters["network"], adapter_name="network")
        network_tokenizer = AutoTokenizer.from_pretrained(NETWORK_BASE_ID, trust_remote_code=True)
        loaded_adapters.append("network")
        print(f"  ✅ Mounted dedicated Qwen2.5 adapter: network")
    except Exception as err:
        print(f"  ℹ️ Dedicated network model skipped or VRAM constrained: {err}")
        print(f"  Network domain analysis will execute with expert forensic prompt on {BASE_MODEL_ID}.")
        loaded_adapters.append("network (prompt-grounded)")

print(f"\n🔥 Model Setup Complete! Active Loaded Adapters: {loaded_adapters}")


## Step 4: Define FastAPI Microservice Endpoints

In [ ]:
import asyncio, json, re, uuid
from fastapi import FastAPI, Header, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Any, Dict, List

# Shared Auth Secret (Must match SHERLOCK_REMOTE_TOKEN in backend .env)
AUTH_TOKEN = os.environ.get("SHERLOCK_REMOTE_TOKEN", "SHERLOCK_SECRET_TOKEN_2026")

# Thread-safe inference lock for atomic adapter switching & text generation
inference_lock = asyncio.Lock()

app = FastAPI(
    title="SherlockAI Remote GPU Inference Bridge",
    version="2.0.0",
    description="Provides GPU-accelerated Qwen multi-LoRA inference for digital forensic agents"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class InferenceRequest(BaseModel):
    domain: str
    evidence_type: str
    evidence: Any
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.1

def verify_auth(x_sherlock_token: Optional[str] = Header(None)):
    if AUTH_TOKEN and x_sherlock_token != AUTH_TOKEN:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid or missing X-Sherlock-Token")

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "hardware": "Kaggle Cloud GPU (Nvidia)",
        "base_model": BASE_MODEL_ID,
        "loaded_adapters": loaded_adapters,
        "peft_enabled": peft_model is not None or network_model is not None
    }

@app.get("/api/v1/adapters")
def list_adapters():
    active = None
    if peft_model is not None:
        active = getattr(peft_model, "active_adapter", None)
    return {
        "loaded_adapters": loaded_adapters,
        "active_adapter": active,
        "available_domains": ["network", "memory", "disk", "logs", "registry", "base_cybersecurity", "supervisor"]
    }

@app.post("/api/v1/inference")
async def run_inference(req: InferenceRequest, x_sherlock_token: Optional[str] = Header(None)):
    verify_auth(x_sherlock_token)
    domain = req.domain.lower()
    
    # Map 'supervisor' request to base_cybersecurity adapter
    target_adapter = "base_cybersecurity" if domain in ["supervisor", "base_cybersecurity"] else domain

    async with inference_lock:
        active_model = inference_model
        active_tok = tokenizer
        adapter_used = "base_model"

        if domain == "network" and network_model is not None:
            active_model = network_model
            active_tok = network_tokenizer
            adapter_used = "network (Qwen2.5-7B LoRA)"
        elif peft_model is not None and target_adapter in loaded_adapters:
            peft_model.set_adapter(target_adapter)
            adapter_used = target_adapter
        elif domain == "network":
            adapter_used = f"{BASE_MODEL_ID} (Domain Specialist)"
        else:
            adapter_used = f"{BASE_MODEL_ID} (Base)"
        
        system_prompt = (
            f"You are the SherlockAI {domain.capitalize()} Forensic Analysis Agent. "
            "Analyze the provided forensic telemetry with zero hallucination. "
            "Return ONLY a valid JSON object matching this schema:\n"
            "{\n"
            '  "classification": "BENIGN" | "SUSPICIOUS" | "MALICIOUS" | "INCONCLUSIVE",\n'
            '  "risk_level": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",\n'
            '  "confidence": 0.0 to 1.0,\n'
            '  "reasoning": "string explaining causal forensic deduction",\n'
            '  "key_evidence": ["specific indicator string", ...],\n'
            '  "mitre_technique": "Txxxx.xxx or N/A"\n'
            "}"
        )
        
        evidence_str = json.dumps(req.evidence, indent=2, default=str) if not isinstance(req.evidence, str) else req.evidence
        user_prompt = f"Domain: {domain}\nEvidence Type: {req.evidence_type}\nTelemetry:\n{evidence_str[:4000]}"
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        prompt_text = active_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = active_tok(prompt_text, return_tensors="pt").to(active_model.device)
        
        with torch.no_grad():
            outputs = active_model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                do_sample=False,
                pad_token_id=active_tok.eos_token_id
            )
        
        generated_text = active_tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Extract JSON from output
        json_match = re.search(r"\{.*\}", generated_text, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group(0))
                parsed["model_info"] = {
                    "provider": "Kaggle GPU",
                    "base_model": BASE_MODEL_ID,
                    "adapter_used": adapter_used
                }
                return parsed
            except Exception:
                pass
        
        return {
            "classification": "INCONCLUSIVE",
            "risk_level": "LOW",
            "confidence": 0.5,
            "reasoning": generated_text.strip(),
            "key_evidence": [],
            "raw_output": generated_text,
            "model_info": {"provider": "Kaggle GPU", "base_model": BASE_MODEL_ID, "adapter_used": adapter_used}
        }

print("✅ FastAPI Microservice Endpoints Registered.")


## Step 5: Launch FastAPI Server & Start Cloudflare Tunnel

In [ ]:
import subprocess, time, threading, uvicorn

# 1. Start Uvicorn in a Background Thread
def start_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)
print(" FastAPI microservice listening on http://127.0.0.1:8000")

# 2. Launch Cloudflare Tunnel
print("🌐 Initializing Cloudflare Tunnel (HTTPS)...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
start_wait = time.time()
while time.time() - start_wait < 30:
    line = cf_proc.stderr.readline()
    if "trycloudflare.com" in line:
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if url_match:
            tunnel_url = url_match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 SHERLOCKAI REMOTE GPU BRIDGE IS LIVE!")
    print(f"🔗 Public URL:  {tunnel_url}")
    print(f"🔑 Auth Token:  {AUTH_TOKEN}")
    print("="*70)
    print("\n👉 Instructions:")
    print("1. Open SherlockAI Web Interface in your browser.")
    print("2. Click the 'AI Inference Settings' badge / modal in the top navigation.")
    print(f"3. Paste this URL into the Remote GPU Bridge field:\n   {tunnel_url}")
    print("4. Click 'Save & Test Connection' -> Tier 2 Remote GPU Accelerated will turn GREEN!")
else:
    print("⚠️ Could not automatically detect Cloudflare URL. Please check Cloudflare output or use pyngrok.")